# 03 · Train, compare, and evaluate

**Phases 4, 5 and 6** of the pipeline.

Trains one YOLOv8 model per dataset — the unprocessed baseline plus the nine
processed variants — then ranks them by validation mAP and evaluates the
winner once on the held-out test set.

This is the notebook that turns ten separate experiments into the single
comparison the assignment asks for.

## 1 · Setup

In [4]:
# ---------------------------------------------------------------------
# Locate the project root.
#
# This notebook may sit in notebook/, or in notebook/<yourname>/, so the
# search walks upwards until it finds the folder containing "data".
# Every path below is built from PROJECT_ROOT, so nothing breaks when the
# notebook is moved or when a teammate runs it on a different machine.
# ---------------------------------------------------------------------
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").is_dir() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

if not (PROJECT_ROOT / "data").is_dir():
    raise SystemExit(
        "Could not find the project root (a folder containing 'data').\n"
        "Set PROJECT_ROOT manually, for example:\n"
        "    PROJECT_ROOT = Path(r'C:/Users/you/PCB-Defect-Inspection')")

RAW_DIR       = PROJECT_ROOT / "data" / "raw"        # Annotations + images
YOLO_DIR      = PROJECT_ROOT / "data" / "yolo"       # generated in notebook 01
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"  # generated in notebook 02
RUNS_DIR      = PROJECT_ROOT / "runs" / "pcb_comparison"

print("Project root :", PROJECT_ROOT)
print("Raw data     :", RAW_DIR, "  exists:", RAW_DIR.is_dir())

Project root : e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection
Raw data     : e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\data\raw   exists: True


In [6]:
import json

import pandas as pd

# ---------------------------------------------------------------------
# The baseline is an ordinary entry in this table, not a special case.
# Without it there is nothing to compare against, and a ranking of nine
# processed variants cannot show whether processing helped at all.
# ---------------------------------------------------------------------
EXPERIMENTS = {"BASE": YOLO_DIR / "data.yaml"}
for key in ["A1", "A2", "A3", "B1", "B2", "B3", "C1", "C2", "C3"]:
    EXPERIMENTS[key] = PROCESSED_DIR / f"pcb_{key}" / "data.yaml"

LABELS = {
    "BASE": "Baseline (no processing)",
    "A1": "Gaussian, light",
    "A2": "Gaussian, medium",
    "A3": "Gaussian, heavy",
    "B1": "Canny edge, weak",
    "B2": "Canny edge, medium",
    "B3": "Canny edge, strong",
    "C1": "Morphological, small",
    "C2": "Morphological, medium",
    "C3": "Morphological, large",
}

# ---------------------------------------------------------------------
# EVERY RUN USES THESE SAME SETTINGS.
#
# If one variant were given more epochs or a larger input size than
# another, the comparison would be measuring that difference instead of
# the image processing. The dataset is the only thing allowed to vary.
# ---------------------------------------------------------------------
MODEL    = "yolov8n.pt"
EPOCHS   = 50
IMGSZ    = 640   # the defects are only 2-4% of the image width, so 640
                  # may be too small to resolve them
BATCH    = 5      # lower this first if you run out of GPU memory
PATIENCE = 20     # stop early if validation stops improving
SEED     = 42     # same seed everywhere, so the comparison is not
                  # measuring random initialisation

RUNS_DIR.mkdir(parents=True, exist_ok=True)

print(f"{'key':<6}{'dataset':<12}exists")
for key, path in EXPERIMENTS.items():
    print(f"{key:<6}{path.parent.name:<12}{path.exists()}")

key   dataset     exists
BASE  yolo        True
A1    pcb_A1      True
A2    pcb_A2      True
A3    pcb_A3      True
B1    pcb_B1      True
B2    pcb_B2      True
B3    pcb_B3      True
C1    pcb_C1      True
C2    pcb_C2      True
C3    pcb_C3      True


## 2 · Train

Ten runs. This is the long part — hours, not minutes.

Already-completed runs are skipped, so you can interrupt this cell and re-run
it later to resume where it stopped.

**Do a short trial first.** Set `EPOCHS = 3` above and run one configuration
to confirm the whole path works before committing to the real thing.

In [7]:
from ultralytics import YOLO


def train_one(key, force=False):
    """Train a single configuration."""
    yaml_path = EXPERIMENTS[key]
    if not yaml_path.exists():
        print(f"SKIP {key}: {yaml_path} not found")
        return

    if (RUNS_DIR / key / "weights" / "best.pt").exists() and not force:
        print(f"SKIP {key}: already trained (pass force=True to redo)")
        return

    print(f"\n{'=' * 68}\nTRAINING {key}: {LABELS[key]}\n{'=' * 68}")

    # A fresh model every time. Continuing from a previous run's weights
    # would let one dataset influence the results of the next.
    model = YOLO(MODEL)
    model.train(
        data=str(yaml_path),
        epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
        patience=PATIENCE, seed=SEED,
        project=str(RUNS_DIR), name=key, exist_ok=True,
        val=True,      # validate after every epoch
        plots=True,
    )


print("train_one() defined. Run the next cell to train everything.")

train_one() defined. Run the next cell to train everything.


In [8]:
# Train the baseline first: if something is wrong, you find out on the
# simplest configuration rather than after nine runs.
for key in EXPERIMENTS:
    train_one(key)

SKIP BASE: already trained (pass force=True to redo)

TRAINING A1: Gaussian, light
New https://pypi.org/project/ultralytics/8.4.137 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.135  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=5, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=e:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\data\processed\pcb_A1\data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hs

## 3 · Collect the results

Ultralytics writes one row per epoch to `results.csv`. The row taken here is
the **best** epoch by mAP50-95, not the last one, which matches the checkpoint
saved as `best.pt`. Reporting the last epoch would penalise any run that
overfitted slightly towards the end.

In [9]:
def collect():
    rows = []
    for key in EXPERIMENTS:
        results_csv = RUNS_DIR / key / "results.csv"
        if not results_csv.exists():
            print(f"  no results yet for {key}")
            continue

        frame = pd.read_csv(results_csv)
        frame.columns = [c.strip() for c in frame.columns]

        metric = "metrics/mAP50-95(B)"
        if metric not in frame.columns:
            print(f"  unexpected columns in {results_csv}")
            continue

        best = frame.loc[frame[metric].idxmax()]
        rows.append({
            "key": key,
            "configuration": LABELS[key],
            "mAP50-95": round(float(best[metric]), 4),
            "mAP50": round(float(best["metrics/mAP50(B)"]), 4),
            "precision": round(float(best["metrics/precision(B)"]), 4),
            "recall": round(float(best["metrics/recall(B)"]), 4),
            "best epoch": int(best["epoch"]),
            "epochs run": int(frame["epoch"].max()),
        })

    if not rows:
        raise SystemExit(
            "No results found. Train the models first (section 2).\n"
            f"Looked in: {RUNS_DIR}")

    return pd.DataFrame(rows).sort_values("mAP50-95", ascending=False)


table = collect()
table.to_csv(RUNS_DIR / "comparison.csv", index=False)
table

,key,configuration,mAP50-95,mAP50,precision,recall,best epoch,epochs run
1,A1,"Gaussian, light",0.4796,0.9232,0.8817,0.8534,27,47
8,C2,"Morphological, medium",0.4693,0.9289,0.9456,0.7901,50,50
6,B3,"Canny edge, strong",0.4684,0.9333,0.9611,0.8091,36,50
9,C3,"Morphological, large",0.4677,0.9382,0.8899,0.8235,45,50
0,BASE,Baseline (no processing),0.4663,0.9475,0.8094,0.9282,50,50
2,A2,"Gaussian, medium",0.4643,0.9359,0.9855,0.7810,45,50
4,B1,"Canny edge, weak",0.4638,0.9209,0.8182,0.8794,32,50
7,C1,"Morphological, small",0.4638,0.9268,0.8965,0.8239,36,50
3,A3,"Gaussian, heavy",0.4600,0.9330,0.8796,0.8768,33,50
5,B2,"Canny edge, medium",0.4563,0.9097,0.8825,0.8104,49,50


In [10]:
# ---------------------------------------------------------------------
# State the outcome plainly, including the awkward one.
# ---------------------------------------------------------------------
if "BASE" not in table["key"].values:
    print("WARNING: the baseline was not trained, so there is nothing to "
          "compare the processed variants against.")
else:
    baseline = table.loc[table["key"] == "BASE", "mAP50-95"].iat[0]
    best = table.iloc[0]

    print(f"Baseline mAP50-95 : {baseline:.4f}")
    print(f"Best variant      : {best['key']} ({best['configuration']}) "
          f"{best['mAP50-95']:.4f}")

    if best["key"] == "BASE":
        # A legitimate outcome that must be reported rather than hidden.
        # A technique that does not help is still a finding, and explaining
        # why belongs in the discussion.
        print("\nNo processed variant beat the baseline. Report this as it "
              "stands: on this dataset the processing did not improve "
              "detection.")
    else:
        change = 100 * (best["mAP50-95"] - baseline) / baseline
        print(f"\nBest variant improves on the baseline by {change:+.1f}% "
              f"relative.")

    beat = table[(table["mAP50-95"] > baseline) & (table["key"] != "BASE")]
    print(f"{len(beat)} of {len(table) - 1} processed variants beat the "
          f"baseline.")

    BEST_KEY = best["key"]
    print(f"\nBEST_KEY = {BEST_KEY!r}")

Baseline mAP50-95 : 0.4663
Best variant      : A1 (Gaussian, light) 0.4796

Best variant improves on the baseline by +2.9% relative.
4 of 9 processed variants beat the baseline.

BEST_KEY = 'A1'


## 4 · Chart for the report

In [11]:
import matplotlib.pyplot as plt

if "table" not in dir():
    raise SystemExit("Run the collect cell in section 3 first.")

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

ordered = table.sort_values("mAP50-95")
colours = ["tab:orange" if k == "BASE" else "tab:blue" for k in ordered["key"]]

axes[0].barh(ordered["configuration"], ordered["mAP50-95"], color=colours)
axes[0].set_xlabel("mAP50-95 (validation)")
axes[0].set_title("Ranked results (orange = baseline)")
axes[0].grid(axis="x", alpha=0.3)

if "BASE" in table["key"].values:
    base_value = table.loc[table["key"] == "BASE", "mAP50-95"].iat[0]
    axes[0].axvline(base_value, color="tab:orange", linestyle="--", alpha=0.7)

for key in EXPERIMENTS:
    results_csv = RUNS_DIR / key / "results.csv"
    if results_csv.exists():
        frame = pd.read_csv(results_csv)
        frame.columns = [c.strip() for c in frame.columns]
        axes[1].plot(frame["epoch"], frame["metrics/mAP50-95(B)"],
                     label=key, linewidth=2 if key == "BASE" else 1)

axes[1].set_xlabel("epoch")
axes[1].set_ylabel("mAP50-95")
axes[1].set_title("Validation curves")
axes[1].legend(fontsize=8, ncol=2)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RUNS_DIR / "comparison.png", dpi=150, bbox_inches="tight")
plt.show()

<Figure size 1500x500 with 2 Axes>

## 5 · Final test evaluation

**This runs once, on the winner only, after all tuning is finished.**

The test set stays untouched during training and selection, which is what
makes this number an unbiased estimate. Evaluating every variant on the test
set and then picking the best would defeat that: the choice would have been
made using the test data, and the reported score would be optimistic.

This is the number that goes in your report.

In [13]:
# BEST_KEY is set by the cell in section 3. Override it here if you want to
# evaluate a different configuration.
if "BEST_KEY" not in dir():
    raise SystemExit("Run section 3 first, or set BEST_KEY manually, "
                     "e.g. BEST_KEY = 'B2'")

weights = RUNS_DIR / BEST_KEY / "weights" / "best.pt"
if not weights.exists():
    raise SystemExit(f"No trained weights at {weights}")

print(f"FINAL TEST EVALUATION: {BEST_KEY} ({LABELS[BEST_KEY]})\n")

model = YOLO(str(weights))
metrics = model.val(data=str(EXPERIMENTS[BEST_KEY]), split="test",
                    project=str(RUNS_DIR), name=f"{BEST_KEY}_test",
                    exist_ok=True)

summary = {
    "configuration": BEST_KEY,
    "description": LABELS[BEST_KEY],
    "test mAP50-95": round(float(metrics.box.map), 4),
    "test mAP50": round(float(metrics.box.map50), 4),
    "test precision": round(float(metrics.box.mp), 4),
    "test recall": round(float(metrics.box.mr), 4),
}

print("Final held-out test result (report this):")
for name, value in summary.items():
    print(f"   {name:<18}{value}")

# Per-class results usually make the most useful discussion: they show
# whether one defect type is dragging the average down.
print("\nPer class (mAP50-95):")
for index, class_name in model.names.items():
    try:
        print(f"   {class_name:<18}{metrics.box.maps[index]:.4f}")
    except (IndexError, TypeError):
        pass

(RUNS_DIR / "test_result.json").write_text(json.dumps(summary, indent=2))
print(f"\nSaved to {RUNS_DIR / 'test_result.json'}")

FINAL TEST EVALUATION: A1 (Gaussian, light)

Ultralytics 8.4.135  Python-3.12.10 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 3050 Laptop GPU, 4096MiB)


Model summary (fused): 73 layers, 3,006,818 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.30.1 ms, read: 197.762.3 MB/s, size: 1369.3 KB)
val: Scanning E:\SCHOOL\DEGY3S1\Image Processing\PCB-Defect-Inspection\data\processed\pcb_A1\labels\test.cache... 152 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 152/152  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 10/10 1.7it/s 5.8s.4ss
                   all        152        520      0.777      0.592      0.627      0.279
          missing_hole         25         85          1      0.888      0.985      0.523
            mouse_bite         25         86      0.806      0.616      0.697      0.327
          open_circuit         26         89      0.538       0.64       0.42       0.19
                 short         25         84      0.606      0.488      0.446      0.101
                  spur         25         85      0.714      0.435      0.536  

---

## What to put in the report

- `runs/pcb_comparison/comparison.csv` — the ranked validation table
- `runs/pcb_comparison/comparison.png` — the ranking chart and curves
- `runs/pcb_comparison/test_result.json` — the final held-out number
- `runs/pcb_comparison/<KEY>/` — per-run curves, confusion matrix, sample
  predictions produced by Ultralytics

Report the negative results too. A configuration that made detection *worse*
is evidence about where a technique stops helping, and a comparative study
that reports only its successes is not a comparative study.

In [14]:
import pandas as pd
t = pd.read_csv(RUNS_DIR / "comparison.csv")
print(t.to_string(index=False))
print("\nspread:", round(t["mAP50-95"].max() - t["mAP50-95"].min(), 4))

 key            configuration  mAP50-95  mAP50  precision  recall  best epoch  epochs run
  A1          Gaussian, light    0.4796 0.9232     0.8817  0.8534          27          47
  C2    Morphological, medium    0.4693 0.9289     0.9456  0.7901          50          50
  B3       Canny edge, strong    0.4684 0.9333     0.9611  0.8091          36          50
  C3     Morphological, large    0.4677 0.9382     0.8899  0.8235          45          50
BASE Baseline (no processing)    0.4663 0.9475     0.8094  0.9282          50          50
  A2         Gaussian, medium    0.4643 0.9359     0.9855  0.7810          45          50
  B1         Canny edge, weak    0.4638 0.9209     0.8182  0.8794          32          50
  C1     Morphological, small    0.4638 0.9268     0.8965  0.8239          36          50
  A3          Gaussian, heavy    0.4600 0.9330     0.8796  0.8768          33          50
  B2       Canny edge, medium    0.4563 0.9097     0.8825  0.8104          49          50

spread: 0